# 05 Silver DLT Pipeline — Integration Layer

**Laag:** Silver / Integration  
**Schema:** `INTEGRATION`  
**Pipeline type:** Lakeflow Declarative Pipeline (DLT)

## Wat doet deze pipeline?

Leest van Bronze (`STAGING_AZURESTORAGE`) via **Change Data Feed** en mergt de
change-events declaratief via **`apply_changes`** naar Silver Streaming Tables.
Bronze-overschrijvingen (full-load mode) verschijnen als `delete_row` + `insert_row`
events in de CDF en worden clean verwerkt — geen Silver-dubbelen, geen pipeline-break
bij een mode-switch in Bronze.  Zie ADR 0002 en `CONTEXT.md §7`.

## Tabellen in deze pipeline (slice #18)

| Tabel | Type | Inhoud |
|---|---|---|
| `INTEGRATION.order_header` | Streaming Table | Gecleaned `order_header` — type-fixes + snake_case |

## Type-fixes (Bronze → Silver)

| Bronze-kolom | Bronze-type | Silver-kolom | Silver-type |
|---|---|---|---|
| `SERVED_TS` | `StringType` | `served_ts` | `TimestampType` |
| `ORDER_TAX_AMOUNT` | `StringType` | `order_tax_amount` | `DecimalType(38,4)` |
| `ORDER_DISCOUNT_AMOUNT` | `StringType` | `order_discount_amount` | `DecimalType(38,4)` |
| `LOCATION_ID` | `DoubleType` | `location_id` | `DecimalType(38,0)` |
| `DISCOUNT_ID` | `StringType` | `discount_id` | `DecimalType(38,0)` nullable |
| `SHIFT_START_TIME` | `IntegerType` (millis) | `shift_start_time` | `StringType` `'HH:mm:ss'` |
| `SHIFT_END_TIME` | `IntegerType` (millis) | `shift_end_time` | `StringType` `'HH:mm:ss'` |

All other columns: same type, renamed to snake_case.
All five Bronze audit columns carry through unchanged.

In [ ]:
%run ./_lib/type_helpers

In [ ]:
%run ./_lib/bronze_cdf

In [ ]:
import dlt
from pyspark.sql import functions as F

## Parameters

DLT reads pipeline parameters from `spark.conf`.  The `catalog` parameter is passed
by the DAB pipeline configuration (`resources/dlt_integration.yml`).

In [ ]:
catalog = spark.conf.get("pipeline.catalog", "DEMO")
bronze_schema = "STAGING_AZURESTORAGE"

print(f"Catalog : {catalog}")
print(f"Bronze  : {catalog}.{bronze_schema}")

## order_header — cleansed Streaming Table

**Pattern:**
1. `dlt.create_streaming_table` declares the target with `apply_changes` semantics.
2. A `@dlt.view` reads Bronze CDF and applies type-fixes + snake_case rename.
3. `dlt.apply_changes` wires the view into the target Streaming Table.

The CDF `_change_type` column drives the MERGE: `insert_row` / `update_postimage`
become upserts; `delete_row` / `update_preimage` drive deletes.  `truncate_row`
events (from full-load overwrites) are handled as deletes by `apply_changes`.

In [ ]:
dlt.create_streaming_table(
    name="order_header",
    comment="Cleansed order_header: type-fixes (CONTEXT.md §7), snake_case columns, CDF+apply_changes from Bronze.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_header_bronze_cdf")
def order_header_bronze_cdf():
    """Read Bronze order_header via CDF and apply type-fixes + snake_case rename.

    Returns a streaming DataFrame with:
    - All business columns renamed to snake_case with type corrections
    - All five Bronze audit columns carried through unchanged
    - CDF metadata columns (_change_type, _commit_version, _commit_timestamp)
      retained for apply_changes routing
    """
    src = read_bronze_cdf(f"{catalog}.{bronze_schema}.order_header")

    return (
        src
        # ----------------------------------------------------------------
        # Business columns — type-fixes (CONTEXT.md §7) + snake_case
        # ----------------------------------------------------------------
        .withColumn("order_id",               F.col("ORDER_ID").cast("decimal(38,0)"))
        .withColumn("truck_id",               F.col("TRUCK_ID").cast("decimal(38,0)"))
        .withColumn("location_id",            cast_decimal_38_0(F.col("LOCATION_ID")))   # Double → Decimal
        .withColumn("customer_id",            F.col("CUSTOMER_ID").cast("decimal(38,0)"))
        .withColumn("discount_id",            cast_decimal_38_0(F.col("DISCOUNT_ID")))   # String → Decimal nullable
        .withColumn("shift_id",               F.col("SHIFT_ID").cast("decimal(38,0)"))
        .withColumn("shift_start_time",       millis_to_hhmmss(F.col("SHIFT_START_TIME")))  # Int millis → HH:mm:ss
        .withColumn("shift_end_time",         millis_to_hhmmss(F.col("SHIFT_END_TIME")))    # Int millis → HH:mm:ss
        .withColumn("order_channel",          F.col("ORDER_CHANNEL"))
        .withColumn("order_ts",               F.col("ORDER_TS"))
        .withColumn("served_ts",              cast_served_ts(F.col("SERVED_TS")))            # String → Timestamp
        .withColumn("order_currency",         F.col("ORDER_CURRENCY"))
        .withColumn("order_amount",           F.col("ORDER_AMOUNT").cast("decimal(38,4)"))
        .withColumn("order_tax_amount",       cast_decimal_38_4(F.col("ORDER_TAX_AMOUNT")))        # String → Decimal
        .withColumn("order_discount_amount",  cast_decimal_38_4(F.col("ORDER_DISCOUNT_AMOUNT")))   # String → Decimal
        .withColumn("order_total",            F.col("ORDER_TOTAL").cast("decimal(38,4)"))
        # ----------------------------------------------------------------
        # Bronze audit columns — carried through unchanged (CONTEXT.md §5)
        # ----------------------------------------------------------------
        .withColumn("_ingestion_timestamp",  F.col("_ingestion_timestamp"))
        .withColumn("_source_system",        F.col("_source_system"))
        .withColumn("_source_file",          F.col("_source_file"))
        .withColumn("_last_modified",        F.col("_last_modified"))
        .withColumn("_pipeline_run_id",      F.col("_pipeline_run_id"))
        # ----------------------------------------------------------------
        # Select only the Silver columns (drops all uppercase Bronze cols
        # and keeps CDF metadata for apply_changes)
        # ----------------------------------------------------------------
        .select(
            # CDF routing metadata
            "_change_type",
            "_commit_version",
            "_commit_timestamp",
            # Business columns (snake_case, type-fixed)
            "order_id",
            "truck_id",
            "location_id",
            "customer_id",
            "discount_id",
            "shift_id",
            "shift_start_time",
            "shift_end_time",
            "order_channel",
            "order_ts",
            "served_ts",
            "order_currency",
            "order_amount",
            "order_tax_amount",
            "order_discount_amount",
            "order_total",
            # Audit columns (Bronze passthrough)
            "_ingestion_timestamp",
            "_source_system",
            "_source_file",
            "_last_modified",
            "_pipeline_run_id",
        )
    )

In [ ]:
dlt.apply_changes(
    target="order_header",
    source="order_header_bronze_cdf",
    keys=["order_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## Demo notes

- **DLT graph view**: The Databricks UI shows `order_header_bronze_cdf` reading from
  `STAGING_AZURESTORAGE.order_header`, and `order_header` as the Streaming Table target.
- **Mode-switch test**: Switching Bronze `order_header` between `full` and `incremental`
  and rerunning the workflow will NOT cause Silver row duplication or pipeline failure.
  Full-load overwrites generate `delete_row` + `insert_row` events in CDF, and
  `apply_changes` (SCD Type 1 MERGE) keeps Silver consistent.
- **Future slices**: `order_detail`, quarantine tables, `sales_line` MV, and
  DLT Expectations will be added to this same pipeline notebook in later slices.